In [1]:
import os
import sys
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

# 프로젝트 루트 디렉토리 설정
os.chdir("/home/youngjins/project/belief_trading")

# lib 하위 경로 추가
sys.path.append("/home/youngjins/project/belief_trading/lib/")
sys.path.append("/home/youngjins/project/belief_trading/lib/abides_jpmc_public")

# 환경 임포트
import gym
import abides_gym

# lib 모듈 임포트
from abides_core import abides
from abides_core.utils import parse_logs_df, ns_date, str_to_ns, fmt_ts
from abides_markets.configs import rmsc04
from lib.utils import print_state, gym_types

In [2]:
config = rmsc04.build_config()

# print key and value
for key, value in config.items():
    # if value is list, make empty dictionary for unique class name (key) and count number of elements (value)
    if isinstance(value, list):
        unique_classes = {}
        for v in value:
            class_name = v.__class__.__name__
            if class_name not in unique_classes:
                unique_classes[class_name] = 0
            unique_classes[class_name] += 1
        for class_name, count in unique_classes.items():
            print(f"{class_name}: {count}")
    else:
        print(f"{key}: {value}")

end_state = abides.run(config)  # simulation 실행

[332] INFO abides Simulation Start Time: 2025-03-17 16:47:57.245121
[332] INFO abides_core.kernel --- Simulation time: 2021-02-05 00:00:00, messages processed: 0, wallclock elapsed: 0.00s ---


seed: 28400233
start_time: 1612483200000000000
stop_time: 1612519201000000000
ExchangeAgent: 1
NoiseAgent: 1000
ValueAgent: 102
AdaptiveMarketMakerAgent: 2
MomentumAgent: 12
agent_latency_model: <abides_core.latency_model.LatencyModel object at 0x7faca64e5460>
default_computation_delay: 50
custom_properties: {'oracle': <abides_markets.oracles.sparse_mean_reverting_oracle.SparseMeanRevertingOracle object at 0x7faca68dc850>}
random_state_kernel: RandomState(MT19937)
stdout_log_level: INFO


[332] INFO abides_core.kernel Event Queue elapsed: 0:00:01.831333, messages: 35,943, messages per second: 19626.7
[332] INFO abides_core.kernel Mean ending value by agent type:
[332] INFO abides_core.kernel NoiseAgent: -176
[332] INFO abides_core.kernel ValueAgent: 3037
[332] INFO abides_core.kernel AdaptivePOVMarketMakerAgent: -86012
[332] INFO abides_core.kernel MomentumAgent: 3146
[332] INFO abides_core.kernel Simulation ending!
[332] INFO abides Simulation End Time: 2025-03-17 16:48:00.138562
[332] INFO abides Time taken to run simulation: 0:00:02.893441


In [3]:
# 주문 orderbook 추출
order_book = end_state["agents"][0].order_books["ABM"]

# L1 주문 조회
L1 = order_book.get_L1_snapshots()
best_bids = pd.DataFrame(L1["best_bids"], columns=["time", "price", "qty"])
best_asks = pd.DataFrame(L1["best_asks"], columns=["time", "price", "qty"])

## All times are in ns from 1970, remove the date component to put them in ns from midnight
best_bids["time"] = best_bids["time"].apply(lambda x: x - ns_date(x))
best_asks["time"] = best_asks["time"].apply(lambda x: x - ns_date(x))

plt.plot(best_bids.time, best_bids.price)
plt.plot(best_asks.time, best_asks.price)

band = 100
plt.ylim(100_000 - band, 100_000 + band)

time_mesh = np.arange(str_to_ns("09:30:00"), str_to_ns("10:10:00"), 1e9 * 60 * 10)
_ = plt.xticks(
    time_mesh, [fmt_ts(time).split(" ")[1] for time in time_mesh], rotation=60
)

[2620] INFO abides Simulation Start Time: 2025-03-17 16:14:29.731660


[2620] INFO abides_core.kernel --- Simulation time: 2021-02-05 00:00:00, messages processed: 0, wallclock elapsed: 0.00s ---
[2620] INFO abides_core.kernel Event Queue elapsed: 0:00:01.631853, messages: 32,948, messages per second: 20190.5
[2620] INFO abides_core.kernel Mean ending value by agent type:
[2620] INFO abides_core.kernel NoiseAgent: -68
[2620] INFO abides_core.kernel ValueAgent: 1392
[2620] INFO abides_core.kernel AdaptivePOVMarketMakerAgent: -44736
[2620] INFO abides_core.kernel MomentumAgent: 1290
[2620] INFO abides_core.kernel Simulation ending!
[2620] INFO abides Simulation End Time: 2025-03-17 16:14:32.390383
[2620] INFO abides Time taken to run simulation: 0:00:02.658723


In [5]:
# gym의 종류 dictionary
gym_type_str = "markets-execution-v0"
gym_type_abb = gym_types[gym_type_str]

env = gym.make(gym_type_str, background_config="rmsc04")
# set the general seed
env.seed(0)

state = env.reset()
print_state(state, gym_type_abb)

[2620] INFO abides_core.kernel --- Simulation time: 2021-02-05 00:00:00, messages processed: 0, wallclock elapsed: 0.00s ---


holdingsPct_t: [0.]
timePct_t: [0.]
differencePct_t: [0.]
imbalance5_t: [0.]
imbalanceAll_t: [0.]
priceImpact_t: [0.]
spread_t: [0.]
directionFeature_t: [2.]
R^k_t: [0.]
R^k_t: [0.]
R^k_t: [0.]


In [ ]:
state, reward, done, info = env.step(0)
print_state(state, gym_type_abb)